# Predial Vision MX - Detección de Edificios en Nicolás Romero

**Entrenamiento completo de DeepLab MobileNet V2 en GPU T4**

Este notebook entrena un modelo de segmentación semántica para detectar edificios
en el municipio de Nicolás Romero, Estado de México, usando:
- **Imagery:** ESRI World Imagery (~1m/pixel)
- **Labels:** Microsoft Building Footprints (138,491 polígonos)
- **Modelo:** DeepLab v3+ con MobileNet V2 (pretrained Pascal VOC)
- **Framework:** TensorFlow 1.15

Tiempo estimado: ~30 min en T4 GPU

**Repo:** github.com/MarxCha/predial-vision-mx

## 0. Verificar GPU

In [ ]:
import tensorflow as tf
print(f'TensorFlow: {tf.__version__}')
print(f'GPU disponible: {tf.test.is_gpu_available()}')
if tf.test.is_gpu_available():
    print(f'GPU: {tf.test.gpu_device_name()}')

## 1. Instalar dependencias

In [ ]:
%%bash
# Instalar Raster Vision y dependencias
pip install -q \
    rasterio shapely rtree pyproj \
    everett==0.9 pluginbase==0.7 "supermercado==0.0.5" \
    "click<7" imageio scikit-learn lxml networkx \
    mask-to-polygons tf-slim==1.1.0 boto3

# Instalar RV desde el commit exacto
pip install -q --no-deps \
    git+https://github.com/azavea/raster-vision.git@9f38cc9#egg=rastervision

# Relajar version pins de RV
python3 -c "
import glob, re
for meta in glob.glob('/opt/conda/lib/python*/site-packages/rastervision-*.egg-info/PKG-INFO') + \
            glob.glob('/opt/conda/lib/python*/site-packages/rastervision-*.dist-info/METADATA'):
    with open(meta) as f: content = f.read()
    content = re.sub(r'(Requires-Dist: \\w+)==[\\d.*]+', r'\\1', content)
    with open(meta, 'w') as f: f.write(content)
    print(f'Relaxed: {meta}')
"

# Instalar TF DeepLab models
git clone -q https://github.com/tensorflow/models.git /opt/tf-models
cd /opt/tf-models && git checkout cbbb2ffcde66e646d4a47628ffe2ece2322b64e8
ln -sf /opt/tf-models/research/deeplab /opt/tf-models/deeplab

# Instalar tippecanoe
apt-get update -qq && apt-get install -y -qq build-essential libsqlite3-dev zlib1g-dev > /dev/null
git clone -q https://github.com/mapbox/tippecanoe.git /tmp/tippecanoe
cd /tmp/tippecanoe && git checkout 1.34.3 && make -j$(nproc) -s && make install -s

echo 'Instalación completa!'

In [ ]:
%%bash
# Patch DeepLab para dataset custom
python3 << 'PYEOF'
import os, glob
for path in glob.glob('/opt/tf-models/research/deeplab/datasets/segmentation_dataset.py') + \
            glob.glob('/opt/tf-models/research/deeplab/datasets/data_generator.py'):
    with open(path) as f: code = f.read()
    if '_DATASETS_INFORMATION' not in code: continue
    if 'import os' not in code: code = 'import os\n' + code
    custom = """    'custom': DatasetDescriptor(
        splits_to_sizes={
            'train': int(os.environ.get('DL_CUSTOM_TRAIN', 0)),
            'validation': int(os.environ.get('DL_CUSTOM_VALIDATION', 0)),
        },
        num_classes=int(os.environ.get('DL_CUSTOM_CLASSES', 3)),
        ignore_label=255,
    ),\n"""
    for marker in ["'ade20k': _ADE20K_INFORMATION,", "'pascal_voc_seg': _PASCAL_VOC_SEG_INFORMATION,"]:
        if marker in code:
            code = code.replace(marker, marker + '\n' + custom)
            break
    with open(path, 'w') as f: f.write(code)
    print(f'Patched: {path}')
    break
PYEOF

# Set PYTHONPATH
echo 'export PYTHONPATH=$PYTHONPATH:/opt/tf-models/research:/opt/tf-models/research/slim:/opt/tf-models/research/deeplab' >> ~/.bashrc

In [ ]:
import os
os.environ['PYTHONPATH'] = os.environ.get('PYTHONPATH', '') + ':/opt/tf-models/research:/opt/tf-models/research/slim:/opt/tf-models/research/deeplab'
os.environ['GDAL_DATA'] = '/usr/share/gdal'
import sys
sys.path.extend(['/opt/tf-models/research', '/opt/tf-models/research/slim', '/opt/tf-models/research/deeplab'])

import rastervision as rv
print('RV OK:', hasattr(rv, 'TF_DEEPLAB'))

## 2. Descargar datos

### Opción A: Subir como Kaggle Dataset (recomendado)
Sube estos archivos como un Kaggle Dataset llamado `predial-vision-nr-data`:
- `nicolas-romero.tif` (245MB) - imagery satelital
- `mexico.mbtiles.gz` (109KB) - vector tiles OSM, O `ms_buildings_nr.geojson` para MS Buildings
- `aois/` - carpeta con train-aoi1.geojson, train-aoi2.geojson, val-aoi1.geojson

### Opción B: Descargar en el notebook (más lento)

In [ ]:
# === CONFIGURACIÓN ===
# Cambiar según cómo subiste los datos:

# Opción A: Kaggle Dataset
# KAGGLE_DATA = '/kaggle/input/predial-vision-nr-data'

# Opción B: Descargar aquí
KAGGLE_DATA = None  # Se descargan abajo

# Directorio de trabajo
WORK_DIR = '/kaggle/working/predial-vision'
os.makedirs(f'{WORK_DIR}/input/nicolas-romero/imagery', exist_ok=True)
os.makedirs(f'{WORK_DIR}/input/nicolas-romero/aois', exist_ok=True)
os.makedirs(f'{WORK_DIR}/input/vector-tiles', exist_ok=True)
os.makedirs(f'{WORK_DIR}/rv', exist_ok=True)

print(f'Work dir: {WORK_DIR}')

In [ ]:
%%bash
WORK_DIR=/kaggle/working/predial-vision

# === DESCARGAR IMAGERY (si no se subió como dataset) ===
if [ ! -f "$WORK_DIR/input/nicolas-romero/imagery/nicolas-romero.tif" ]; then
    echo 'Descargando imagery ESRI World Imagery...'
    python3 << 'PYEOF'
import math, os, urllib.request, concurrent.futures, json

bbox_west, bbox_south, bbox_east, bbox_north = -99.36, 19.56, -99.26, 19.64
zoom = 17
work_dir = '/kaggle/working/predial-vision'
tiles_dir = os.path.join(work_dir, 'input/nicolas-romero/imagery/tiles')
os.makedirs(tiles_dir, exist_ok=True)

def lat_lon_to_tile(lat, lon, zoom):
    n = 2 ** zoom
    x = int((lon + 180.0) / 360.0 * n)
    lat_rad = math.radians(lat)
    y = int((1.0 - math.log(math.tan(lat_rad) + 1.0/math.cos(lat_rad)) / math.pi) / 2.0 * n)
    return x, y

x_min, y_min = lat_lon_to_tile(bbox_north, bbox_west, zoom)
x_max, y_max = lat_lon_to_tile(bbox_south, bbox_east, zoom)
total = (x_max - x_min + 1) * (y_max - y_min + 1)
print(f'Descargando {total} tiles...')

def download_tile(args):
    x, y = args
    tp = os.path.join(tiles_dir, f'{zoom}_{x}_{y}.jpg')
    if os.path.exists(tp) and os.path.getsize(tp) > 100: return True
    url = f'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{zoom}/{y}/{x}'
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'PredialVision/1.0'})
        with urllib.request.urlopen(req, timeout=30) as resp:
            with open(tp, 'wb') as f: f.write(resp.read())
        return True
    except: return False

tiles = [(x, y) for y in range(y_min, y_max+1) for x in range(x_min, x_max+1)]
ok = 0
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as ex:
    for r in ex.map(download_tile, tiles):
        if r: ok += 1
print(f'Descargados: {ok}/{total}')

# Crear world files y merge
def tile_to_lat_lon(x, y, zoom):
    n = 2 ** zoom
    lon = x / n * 360.0 - 180.0
    lat_rad = math.atan(math.sinh(math.pi * (1 - 2 * y / n)))
    return math.degrees(lat_rad), lon

tile_files = []
for y in range(y_min, y_max+1):
    for x in range(x_min, x_max+1):
        tp = os.path.join(tiles_dir, f'{zoom}_{x}_{y}.jpg')
        if not os.path.exists(tp) or os.path.getsize(tp) < 100: continue
        lat_top, lon_left = tile_to_lat_lon(x, y, zoom)
        lat_bottom, lon_right = tile_to_lat_lon(x+1, y+1, zoom)
        pw = (lon_right - lon_left) / 256
        ph = (lat_bottom - lat_top) / 256
        with open(tp.replace('.jpg', '.jgw'), 'w') as f:
            f.write(f'{pw}\n0.0\n0.0\n{ph}\n{lon_left+pw/2}\n{lat_top+ph/2}\n')
        tile_files.append(tp)

filelist = os.path.join(work_dir, 'input/nicolas-romero/imagery/tile_list.txt')
with open(filelist, 'w') as f:
    for t in tile_files: f.write(t + '\n')
print(f'World files creados: {len(tile_files)}')
PYEOF

    # Merge con GDAL
    gdalbuildvrt -input_file_list $WORK_DIR/input/nicolas-romero/imagery/tile_list.txt $WORK_DIR/input/nicolas-romero/imagery/tiles.vrt
    gdal_translate -of GTiff -co COMPRESS=LZW -co TILED=YES \
        $WORK_DIR/input/nicolas-romero/imagery/tiles.vrt \
        $WORK_DIR/input/nicolas-romero/imagery/nicolas-romero.tif
    gdal_edit.py -a_srs EPSG:4326 $WORK_DIR/input/nicolas-romero/imagery/nicolas-romero.tif
    echo 'Imagery lista!'
    ls -lh $WORK_DIR/input/nicolas-romero/imagery/nicolas-romero.tif
fi

In [ ]:
%%bash
WORK_DIR=/kaggle/working/predial-vision

# === DESCARGAR OSM BUILDINGS + CREAR MBTILES ===
if [ ! -f "$WORK_DIR/input/vector-tiles/mexico.mbtiles.gz" ]; then
    echo 'Descargando buildings de OSM...'
    curl -s 'https://overpass-api.de/api/interpreter' \
        --data-urlencode 'data=[out:json][timeout:120];(way[building](19.56,-99.36,19.64,-99.26);relation[building](19.56,-99.36,19.64,-99.26););out geom;' \
        -o $WORK_DIR/input/nicolas-romero/osm_buildings.json

    python3 << 'PYEOF'
import json
with open('/kaggle/working/predial-vision/input/nicolas-romero/osm_buildings.json') as f:
    data = json.load(f)
features = []
for el in data.get('elements', []):
    if el['type'] == 'way' and 'geometry' in el:
        coords = [[pt['lon'], pt['lat']] for pt in el['geometry']]
        if coords[0] != coords[-1]: coords.append(coords[0])
        features.append({
            'type': 'Feature',
            'properties': {'@id': f"way/{el['id']}", 'building': el.get('tags',{}).get('building','yes')},
            'geometry': {'type': 'Polygon', 'coordinates': [coords]}
        })
with open('/kaggle/working/predial-vision/input/nicolas-romero/buildings.geojson', 'w') as f:
    json.dump({'type': 'FeatureCollection', 'features': features}, f)
print(f'{len(features)} buildings')
PYEOF

    tippecanoe \
        --output=$WORK_DIR/input/vector-tiles/mexico.mbtiles \
        --layer=osm --minimum-zoom=12 --maximum-zoom=14 \
        --no-feature-limit --no-tile-size-limit --force \
        $WORK_DIR/input/nicolas-romero/buildings.geojson
    gzip -k $WORK_DIR/input/vector-tiles/mexico.mbtiles
    echo 'Vector tiles listos!'
fi

In [ ]:
%%bash
WORK_DIR=/kaggle/working/predial-vision

# === CREAR AOIs ===
python3 << 'PYEOF'
import json, os
aoi_dir = '/kaggle/working/predial-vision/input/nicolas-romero/aois'
os.makedirs(aoi_dir, exist_ok=True)
aois = {
    'train-aoi1.geojson': [[-99.335,19.595],[-99.315,19.595],[-99.315,19.580],[-99.335,19.580],[-99.335,19.595]],
    'train-aoi2.geojson': [[-99.310,19.590],[-99.290,19.590],[-99.290,19.575],[-99.310,19.575],[-99.310,19.590]],
    'val-aoi1.geojson': [[-99.325,19.575],[-99.305,19.575],[-99.305,19.560],[-99.325,19.560],[-99.325,19.575]],
}
for name, coords in aois.items():
    geojson = {'type':'FeatureCollection','features':[{'type':'Feature','properties':{},'geometry':{'type':'Polygon','coordinates':[coords]}}]}
    with open(os.path.join(aoi_dir, name), 'w') as f: json.dump(geojson, f)
    print(f'  {name}')
print('AOIs creados')
PYEOF

## 3. Configurar y correr el experimento

### Clonar el repo y configurar paths

In [ ]:
%%bash
cd /kaggle/working
if [ ! -d predial-vision-mx ]; then
    git clone https://github.com/MarxCha/predial-vision-mx.git
fi

# Ajustar paths para Kaggle (sin Docker)
cd predial-vision-mx
sed -i "s|local_rv_root = '/opt/data/rv'|local_rv_root = '/kaggle/working/predial-vision/rv'|" idb/experiment.py
sed -i "s|local_data_root = '/opt/data/input'|local_data_root = '/kaggle/working/predial-vision/input'|" idb/experiment.py

echo 'Repo configurado para Kaggle'

### Correr entrenamiento completo (150K steps, ~30 min en T4)

In [ ]:
%%bash
export PYTHONPATH=$PYTHONPATH:/opt/tf-models/research:/opt/tf-models/research/slim:/opt/tf-models/research/deeplab
export GDAL_DATA=/usr/share/gdal

cd /kaggle/working/predial-vision-mx

# Entrenamiento completo: 150,000 steps, batch_size 8
# En T4 GPU esto toma ~30 min
# Para test rápido, agregar: -a test True
rastervision run local -e idb.experiment 2>&1 | tee /kaggle/working/training.log | tail -5

## 4. Inspeccionar resultados

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show

WORK_DIR = '/kaggle/working/predial-vision'

# Cargar eval.json
eval_path = f'{WORK_DIR}/rv/eval/multi-city/eval.json'
try:
    with open(eval_path) as f:
        eval_data = json.load(f)
    print('=== Métricas de Evaluación ===')
    for scene in eval_data.get('per_scene', {}):
        print(f'\nEscena: {scene}')
        for metric in eval_data['per_scene'][scene]:
            print(f"  {metric['class_name']}: F1={metric['f1']:.3f}, Precision={metric['precision']:.3f}, Recall={metric['recall']:.3f}")
except FileNotFoundError:
    print('eval.json no generado aún')

In [ ]:
# Visualizar predicciones sobre imagery
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# Imagery original
with rasterio.open(f'{WORK_DIR}/input/nicolas-romero/imagery/nicolas-romero.tif') as src:
    img = src.read([1,2,3])
    show(img, ax=axes[0], title='Imagery Satelital')

# Predicciones
pred_path = f'{WORK_DIR}/rv/predict/multi-city/nicolas_romero.tif'
try:
    with rasterio.open(pred_path) as src:
        pred = src.read(1)
        axes[1].imshow(pred == 1, cmap='Oranges', alpha=0.8)
        axes[1].set_title('Predicciones (Building = naranja)')
        
        # Overlay
        show(img, ax=axes[2])
        axes[2].imshow(pred == 1, cmap='Oranges', alpha=0.4)
        axes[2].set_title('Overlay')
except FileNotFoundError:
    print('Predicciones no generadas aún')

plt.tight_layout()
plt.savefig('/kaggle/working/predicciones_nicolas_romero.png', dpi=150)
plt.show()

# Stats
if 'pred' in dir():
    unique, counts = np.unique(pred, return_counts=True)
    total = pred.size
    print('\n=== Distribución de predicciones ===')
    labels = {0: 'NODATA', 1: 'Building', 2: 'Background'}
    for v, c in zip(unique, counts):
        print(f"  {labels.get(v, v)}: {c:,} pixels ({100*c/total:.1f}%)")

## 5. Descargar resultados

In [ ]:
%%bash
WORK_DIR=/kaggle/working/predial-vision

echo '=== Archivos generados ==='
echo ''
echo '--- Modelo ---'
ls -lh $WORK_DIR/rv/train/multi-city/model 2>/dev/null
ls -lh $WORK_DIR/rv/bundle/multi-city/predict_package.zip 2>/dev/null
echo ''
echo '--- Predicciones ---'
ls -lh $WORK_DIR/rv/predict/multi-city/nicolas_romero.tif 2>/dev/null
ls -lh $WORK_DIR/rv/predict/multi-city/*polygons* 2>/dev/null
echo ''
echo '--- Evaluación ---'
cat $WORK_DIR/rv/eval/multi-city/eval.json 2>/dev/null | python3 -m json.tool | head -30

# Empaquetar para descarga
cd $WORK_DIR/rv
tar czf /kaggle/working/predial-vision-results.tar.gz \
    bundle/multi-city/predict_package.zip \
    predict/multi-city/*.tif \
    predict/multi-city/*polygons* \
    eval/multi-city/eval.json \
    train/multi-city/model \
    2>/dev/null

echo ''
echo 'Descarga: predial-vision-results.tar.gz'
ls -lh /kaggle/working/predial-vision-results.tar.gz